In [2]:
%pip -q install google-genai

In [3]:
import os
from google.colab import userdata

os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')

In [4]:
from google import genai

client = genai.Client()

MODEL_ID = "gemini-2.0-flash"

In [5]:
# Pergunta ao Gemini uma informação mais recente que seu conhecimento

from IPython.display import HTML, Markdown

# Perguntar pro modelo quando é a próxima imersão de IA ###############################################
resposta = client.models.generate_content(
    model=MODEL_ID,
    contents='Quando é a próxima Imersão IA com Google Gemini da Alura?',
)

# Exibe a resposta na tela
display(Markdown(f"Resposta:\n {resposta.text}"))

Resposta:
 A Alura ainda não divulgou as datas para uma nova edição da Imersão IA com Google Gemini. A última edição ocorreu em fevereiro de 2024.

Para ficar por dentro das novidades e não perder a próxima edição, você pode fazer o seguinte:

*   **Acompanhe as redes sociais da Alura:** Fique de olho nos perfis da Alura no [Instagram](https://www.instagram.com/aluraonline/), [LinkedIn](https://www.linkedin.com/school/aluraonline/), [YouTube](https://www.youtube.com/aluraonline) e outras plataformas.
*   **Inscreva-se na newsletter da Alura:** Assim, você receberá as novidades diretamente no seu e-mail.
*   **Visite regularmente o site da Alura:** Verifique a página de cursos e eventos para atualizações.

Assim que a Alura divulgar as datas da próxima Imersão IA com Google Gemini, você ficará sabendo!

In [6]:
#Pergunta ao Gemini uma informação utilizando a busca do Google como contexto

response = client.models.generate_content(
    model=MODEL_ID,
    contents='Quando é a próxima Imersão IA com Google Gemini da Alura?',
    config={"tools":[{"google_search":{}}]}
)

# Exibe a resposta na tela
display(Markdown(f"Resposta:\n {response.text}"))

Resposta:
 A Imersão IA com Google Gemini da Alura mais recente aconteceu entre os dias 12 e 16 de maio de 2025. As inscrições estiveram abertas até o dia 11 de maio de 2025.


In [7]:
#Exibe a busca
print(f"Busca realizada: {response.candidates[0].grounding_metadata.web_search_queries}")
# Exibe as URLs nas quais ele se baseou
print(f"Páginas utilizadas na resposta: {', '.join([site.web.title for site in response.candidates[0].grounding_metadata.grounding_chunks])}")
print()
display(HTML(response.candidates[0].grounding_metadata.search_entry_point.rendered_content))

Busca realizada: ['Alura Imersão IA Google Gemini']
Páginas utilizadas na resposta: youtube.com



In [8]:
!pip install -q google-adk

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 18.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 232.1/232.1 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.2/95.2 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 217.1/217.1 kB 13.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 334.1/334.1 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.1/125.1 kB 7.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.8/65.8 kB 3.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.0/119.0 kB 5.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.9/194.9 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.5/62.5 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.3/103.3 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.0

In [10]:
# Função auxiliar para enviar mensagem ao agente
def call_agent(agent: Agent, message_text: str) -> str:
    session_service = InMemorySessionService()
    session = session_service.create_session(app_name=agent.name, user_id="user1", session_id="session1")
    runner = Runner(agent=agent, app_name=agent.name, session_service=session_service)
    content = types.Content(role="user", parts=[types.Part(text=message_text)])

    final_response = ""
    for event in runner.run(user_id="user1", session_id="session1", new_message=content):
        if event.is_final_response():
            for part in event.content.parts:
                if part.text is not None:
                    final_response += part.text
                    final_response += "\n"
    return final_response

In [11]:
# Função auxiliar para exibir texto formatado em Markdown no Colab
def to_markdown(text):
  text = text.replace('•', '  *')
  return Markdown(textwrap.indent(text, '> ', predicate=lambda _: True))

In [24]:
!pip install serpapi
import os
os.environ["SERPAPI_API_KEY"] = "GOOGLE_API_KEY"
from serpapi import search

In [40]:
!git init
!git add .
!git add chatbot empresas.ipynb
!git commit -m "Mensagem inicial do commit"
!git config --global user.email "ronaldoferreiraramosp@gmail.com"
!git config --global user.name "ronaldoferreira156"
!git remote add origin https://github.com/ronaldoferreira156/chatbot-assistente-pequenos-neg-cios/tree/main.git
!git push -u origin main



Reinitialized existing Git repository in /content/.git/
fatal: pathspec 'chatbot' did not match any files
On branch master
nothing to commit, working tree clean
error: remote origin already exists.
error: src refspec main does not match any
error: failed to push some refs to 'http://github.com/ronaldoferreira156/chatbot-assistente-pequenos-neg-cios/tree/main'


In [29]:
import os
from google.colab import userdata
from google import genai
from google.adk.agents import Agent
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types
from datetime import date
import textwrap
from IPython.display import display, Markdown
import warnings
from serpapi import search  # Importando a função 'search' do serpapi

warnings.filterwarnings("ignore")

# Configuração da chave da API do Google (para o modelo Gemini)
os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')
client = genai.Client()
MODEL_ID = "gemini-2.0-flash"

# Função auxiliar para exibir texto formatado em Markdown
def to_markdown(text):
    text = text.replace('•', '  *')
    return Markdown(textwrap.indent(text, '> ', predicate=lambda _: True))

# Função auxiliar para enviar mensagem ao agente
def call_agent(agent: Agent, message_text: str) -> str:
    session_service = InMemorySessionService()
    session = session_service.create_session(app_name=agent.name, user_id="user1", session_id="session1")
    runner = Runner(agent=agent, app_name=agent.name, session_service=session_service)
    content = types.Content(role="user", parts=[types.Part(text=message_text)])

    final_response = ""
    for event in runner.run(user_id="user1", session_id="session1", new_message=content):
        if event.is_final_response():
            for part in event.content.parts:
                if part.text is not None:
                    final_response += part.text
                    final_response += "\n"
    return final_response

# --- Ferramentas ---

# Ferramenta simulada para informações locais
def obter_info_local(location: str) -> str:
    """Fornece informações genéricas sobre recursos locais para pequenos negócios."""
    return f"Para encontrar informações específicas para {location}, procure por órgãos como a prefeitura local (secretaria de desenvolvimento econômico), associações comerciais, e o SEBRAE de sua região. Eles geralmente oferecem suporte, informações sobre licenças e cadastros municipais, e podem ter diretórios de fornecedores locais."

# Ferramenta de cálculo de imposto MEI (simplificada)
def calcular_das_mei(faturamento_mensal: float, tipo_atividade: str) -> str:
    """Calcula uma estimativa simplificada do DAS para MEI."""
    taxa_servicos = 0.05  # Exemplo de taxa para serviços
    taxa_comercio_industria = 0.01  # Exemplo de taxa para comércio e indústria
    valor_inss = 66.00  # Valor de INSS em 2023 (pode precisar ser atualizado)
    valor_iss_servicos = 5.00  # Exemplo de ISS para serviços
    valor_icms_comercio_industria = 1.00  # Exemplo de ICMS para comércio e indústria

    if tipo_atividade.lower() == "serviços":
        das_estimado = valor_inss + valor_iss_servicos
        return f"Para serviços, o DAS MEI estimado é de R$ {das_estimado:.2f} (INSS + ISS)."
    elif tipo_atividade.lower() in ["comércio", "indústria"]:
        das_estimado = valor_inss + valor_icms_comercio_industria
        return f"Para comércio ou indústria, o DAS MEI estimado é de R$ {das_estimado:.2f} (INSS + ICMS)."
    else:
        return "Não foi possível estimar o DAS com o tipo de atividade fornecida."

# Define o Agente com a instrução para realizar buscas diretamente
agente_negocios = Agent(
    name="AssistentePequenosNegocios",
    model=MODEL_ID,
    description="Agente de suporte para fornecer informações e recursos a pequenos negócios.",
    tools=[obter_info_local, calcular_das_mei],
    instruction="""
    Você é um assistente amigável e informativo para pequenos empreendedores em todo o Brasil.
    Você tem acesso às seguintes ferramentas:
    - obter_info_local(location: str): Para obter informações genéricas sobre recursos locais para uma determinada cidade ou região.
    - calcular_das_mei(faturamento_mensal: float, tipo_atividade: str): Para calcular uma estimativa simplificada do Documento de Arrecadação do Simples Nacional (DAS) para MEI.

    Quando o usuário fizer uma pergunta que claramente requer informações da internet (como 'quais os concorrentes de petshop em Vitória da Conquista' ou 'dicas de marketing digital'), você pode usar suas capacidades de busca para tentar encontrar essa informação e apresentar um resumo conciso e os links relevantes, se disponíveis.

    Para usar 'obter_info_local', peça ao usuário a localização.
    Para usar 'calcular_das_mei', peça o faturamento mensal e o tipo de atividade.

    Ao responder:
    - Seja claro e conciso.
    - Forneça informações gerais sobre registro (CNPJ) e licenças, incentivando a consulta aos órgãos locais.
    - Mencione o SEBRAE (www.sebrae.com.br).
    - Apresente ferramentas do Google para negócios.
    - Ofereça dicas de marketing digital.
    - Formate suas respostas em Markdown.
    - Se a pergunta não for relacionada a pequenos negócios, informe que não pode ajudar.

    Mantenha um tom encorajador.
    """
)

# Loop de conversação (sem código de busca externa)
print("Bem-vindo ao Assistente para Pequenos Negócios!")
print("Como posso te ajudar hoje? (Digite 'sair' para encerrar)")

while True:
    pergunta_usuario = input("Você: ")
    if pergunta_usuario.lower() == "sair":
        print("Obrigado por usar o assistente! Até a próxima!")
        break
    else:
        resposta_agente = call_agent(agente_negocios, pergunta_usuario)
        display(to_markdown(resposta_agente))

Bem-vindo ao Assistente para Pequenos Negócios!
Como posso te ajudar hoje? (Digite 'sair' para encerrar)
Você: sair
Obrigado por usar o assistente! Até a próxima!
